In [105]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score, precision_score, root_mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split

In [92]:
def cross_validate_regression(training_data, training_labels):
    model = xgb.XGBRegressor(objective="reg:squarederror", random_state=42)
    parameters = {
        'subsample': [0.5, 0.7, 1],
        'max_depth': [3, 6, 10],
        'gamma': [0, 1, 5],
        'eta': [0.1, 0.3, 0.5]
    }
    validation = GridSearchCV(estimator=model, param_grid=parameters, n_jobs=-1, verbose=4)
    validation.fit(training_data, training_labels)
    return validation.best_estimator_

In [93]:
# This one is for binary classification
def cross_validate_classification(training_data, training_labels):
    model = xgb.XGBClassifier(objective="binary:logistic", random_state=42)
    parameters = {
        'subsample': [0.5, 0.7, 1],
        'max_depth': [3, 6, 10],
        'gamma': [0, 1, 5],
        'eta': [0.1, 0.3, 0.5]
    }
    validation = GridSearchCV(estimator=model, param_grid=parameters, n_jobs=-1, verbose=4)
    validation.fit(training_data, training_labels)
    
    return validation.best_estimator_

In [94]:
# This one is for multi class
def cross_validate_classification_multi(training_data, training_labels):
    model = xgb.XGBClassifier(objective="multi:softmax", random_state=42)
    parameters = {
        'subsample': [0.5, 0.7, 1],
        'max_depth': [3, 6, 10],
        'gamma': [0, 1, 5],
        'eta': [0.1, 0.3, 0.5]
    }
    validation = GridSearchCV(estimator=model, param_grid=parameters, n_jobs=-1, verbose=4)
    validation.fit(training_data, training_labels)
    
    return validation.best_estimator_

In [95]:
def stats(y_test, y_pred):
    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred)
    print(f"Accuracy: {acc}")
    print(f"Recall: {rec}")
    print(f"Precision: {pre}")
    print(confusion_matrix(y_test, y_pred))
    return

In [96]:
# load data set Bank
data = pd.read_csv("bank.csv")
# Replace unknown values with NaN
data.replace("unknown", np.nan, inplace=True)
info = data.drop(columns="deposit")
# Change categorical columns into one-hot encoding
categorical = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome"]
info = pd.get_dummies(info, columns=categorical)
# Adjust categorical label into binary
encoder = LabelEncoder()
labels = encoder.fit_transform(data["deposit"])

X_train, X_test, y_train, y_test = train_test_split(info, labels, test_size=0.2, random_state=42, stratify=labels)



In [97]:
model = cross_validate_classification(X_train, y_train)

Fitting 5 folds for each of 81 candidates, totalling 405 fits


In [98]:
y_pred = model.predict(X_test)

In [99]:
print("Bank deposit classification stats:")
stats(y_test, y_pred)

Bank deposit classification stats:
Accuracy: 0.8593819973130318
Recall: 0.8799621928166351
Precision: 0.832737030411449
[[988 187]
 [127 931]]


In [100]:
data = pd.read_csv("Student_performance.csv")
info = data.drop(columns="Performance Index")
categorical = ["Extracurricular Activities"]
info = pd.get_dummies(info, columns=categorical)
labels = data["Performance Index"]

X_train, X_test, y_train, y_test = train_test_split(info, labels, test_size=0.2, random_state=42)

In [101]:
model = cross_validate_regression(X_train, y_train)

Fitting 5 folds for each of 81 candidates, totalling 405 fits


In [102]:
y_pred = model.predict(X_test)

In [110]:
def stats_regression(y_pred, y_test):
    rmse = root_mean_squared_error(y_pred, y_test)
    r2 = r2_score(y_pred, y_test)
    print(f"Root Mean Squared Error: {rmse}")
    print(f"Coefficient of Determination: {r2}")
    return

In [111]:
print("Student Performance Regression Stats:")
stats_regression(y_pred, y_test)

Student Performance Regression Stats:
Root Mean Squared Error: 2.070702704086782
Coefficient of Determination: 0.988090657197757
